In [10]:
import os
from pathlib import Path

import sqlglot
from sqlglot import exp, expressions

from src.utils.file_utils import parse_file_name
from src.migration.com.decomposer import ComSqlDecomposer, ComDecomposerWriter
from src.migration.com.metadata import ComMetadataProcessor
from src.migration.com.generator import ComPySparkGenerator
from src.paths import *

In [11]:
USERNAME

'ext_giadung'

In [12]:
from src.utils.source_rule_loader import load_all_source_rules

# input_file = PROJECT_ROOT / "docs" / "datalake_old" /"dml" / "com_r_k2_cif_alias.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "com" /  "com_r_mhbos_m_client_crs.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "com" /  "com_t_mhbos_m_client.sql"

# input_file = DATALAKE_SCRIPT_DIR /"dml" / "com" /  "com_t_lmskibb2_tbl_counterparty.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_contact.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_customer.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_risk_profile.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_customer_employment.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_customer_employment.sql"


output_root = PROJECT_ROOT / "output" / "migration"


def get_file_path(script_name):
    input_file = DATALAKE_SCRIPT_DIR /"dml" / "com" /  f"{script_name}.sql"

    file_name = os.path.basename(input_file).replace('.sql', '')
    layer, sub_layer, source_name, base_table = parse_file_name(input_file)

    all_source_rules = load_all_source_rules()
    source_rules = load_all_source_rules()[source_name] if source_name in all_source_rules else all_source_rules['default']

    print(f"File gốc tại: {input_file}")
    return file_name, input_file, source_rules

In [18]:
from jinja.environment import render_template


def run_migration_pipeline(script_config):
    script_name = script_config[0]
    model = script_config[1]

    file_name, input_file, source_rules = get_file_path(script_name)
    # ==========================================
    # BƯỚC 1: BÓC TÁCH SQL (DECOMPOSER)
    # ==========================================
    decomposer = ComSqlDecomposer(source_rules)
    decomposed_script = decomposer.decompose(input_file)

    # Ghi file sub-SQL ra ổ đĩa
    writer = ComDecomposerWriter()
    writer.write(decomposed_script, output_root / file_name)
    print(f"✅ [Bước 1] Đã bóc tách thành các block tại: {output_root / file_name / 'processing_steps'}")

    # ==========================================
    # BƯỚC 2: XỬ LÝ METADATA (PROCESSOR)
    # ==========================================
    processor = ComMetadataProcessor(source_rules)

    try:
        # Hàm này sẽ phân tích AST, tự động tìm file DDL và trích xuất Schema
        pipeline_config = processor.process(decomposed_script, input_file)

        # Ghi file YAML
        metadata_output_dir = output_root / file_name / "metadata"
        processor.write_yaml(pipeline_config, metadata_output_dir)

        print(f"✅ [Bước 2] Đã xử lý Metadata thành công!")
        print(f"   -> Model nhận diện được: Model {pipeline_config['model_type']}")
        print(f"   -> Khóa (Key) nhận diện được: {pipeline_config['primary_key']['logical_primary_key']}")
        print(f"   -> File YAML đã lưu tại: {metadata_output_dir / (file_name + '.yaml')}")

    except FileNotFoundError as e:
        print(f"❌ [Lỗi Bước 2]: {e}")
        print("💡 Gợi ý: Hãy đảm bảo bạn có file DDL tương ứng tại `docs/datalake_old/ddl/com_r_k2_cif_alias.sql` hoặc cùng thư mục `dml/` để hàm trích xuất Schema hoạt động!")


    print("==========================================")
    print(" BƯỚC 3: SINH CODE PYSPARK (GENERATOR)")
    print("==========================================")

    with open(output_root / file_name / "metadata" / (file_name + '.yaml'), 'r') as f:
        pipeline_config = yaml.safe_load(f)

    generator = ComPySparkGenerator(source_rules, output_mode="simple")
    ddl_context, dml_context = generator.generate(pipeline_config, output_root / file_name)


    # DDL
    template_name = f"model_{model}/com_t_ddl.jinja"
    ddl_sql = render_template(template_name, ddl_context, template_dir=generator.template_dir)

    output_dir = PROJECT_ROOT / "output" / "migration" / "ddl" / "com"
    output_dir.mkdir(parents=True, exist_ok=True)

    # Use a descriptive name for the output file
    output_filename = f"{pipeline_config['layer'].lower()}_{pipeline_config['target_table_name'].lower()}.sql"
    ddl_file = output_dir / output_filename

    ddl_file.write_text(ddl_sql, encoding="utf-8")
    # print(f"Generated DDL from '{template_name}' at {ddl_file}")

    # DML
    try:
        template_name = f"model_{model}/com_t_simple_dml.jinja"
        dml_sql = render_template(template_name, dml_context, template_dir=generator.template_dir)
    except:
        template_name = f"model_{model}/com_t_dml_withpartition.jinja"
        dml_sql = render_template(template_name, dml_context, template_dir=generator.template_dir)


    output_dir = PROJECT_ROOT / "output" / "migration" / "dml" / "com"
    output_dir.mkdir(parents=True, exist_ok=True)

    # Use a descriptive name for the output file
    output_filename = f"{pipeline_config['layer'].lower()}_{pipeline_config['target_table_name'].lower()}.py"
    dml_file = output_dir / output_filename

    dml_file.write_text(dml_sql, encoding="utf-8")
    # print(f"Generated dml from '{template_name}' at {dml_file}")



    print("🎉 Hoàn tất toàn bộ Pipeline!")
    return ddl_context, dml_context


In [20]:

table_list = [
    # "lmskibb2_tbl_counterparty",
    # "smfkibb_tbl_v3_cache_accountmarginposition",
    # "smfkibb_tbl_security",
    # "smfkibb_tbl_policytype",
    # "smfkibb_tbl_account",
    # "sblkibb_tbl_typeofbusiness",
    # "sblkibb_tbl_title",
    # "sblkibb_tbl_residencystatus",
    # "sblkibb_tbl_race",
    # "sblkibb_tbl_occupation",
    # "sblkibb_tbl_maritalstatus",
    # "sblkibb_tbl_counterpartyclass",
    # "sblkibb_tbl_counterparty",
    # "sblkibb_tbl_account",
    # "sbl_tbl_einvoicing_clientdata",
    # "m21_race",
    # "m21_occupation",
    # "m21_currency",
    # "m21_country",
    # "m21_clienttype",
    # "m21_bankaccount",
    # "m21_aetype"
    # "m21_natureofbusiness"


    # ["general_country", "1"],
    # ["toms_eagentsb", "5b"],
    # ["toms_eagentsb_iuta", "5b"],
    # ["m21_o_trx", "5b"],
    # ["m21_a_trx", "5b"],
    # ["kdi_trx", "5b"],
    # ["guava_trx_inc", "5b"],
    # ["mhbos_ecmit029r", "2b"],
    # ["mhbos_ecmit028r", "2b"],
    # ["m21_cashmovement", "5b"],
    ["m21_countrystate", "2a"],
]

for table in table_list:
    try:
        run_migration_pipeline([f"com_t_{table[0]}", table[1]])
    except:
        run_migration_pipeline([f"com_r_{table[0]}", table[1]])
        # try:
        #     run_migration_pipeline([f"com_r_{table[0]}", table[1]])
        # except:
        #     _, input_file, _ = get_file_path(f"com_t_{table}")
        #     input_file.touch(exist_ok=True)
        #     run_migration_pipeline([f"com_t_{table[0]}", table[1]])



File gốc tại: C:\Users\ext_giadung\projects\datalake-script\dml\com\com_t_m21_countrystate.sql
File gốc tại: C:\Users\ext_giadung\projects\datalake-script\dml\com\com_r_m21_countrystate.sql
✅ [Bước 1] Đã bóc tách thành các block tại: C:\Users\ext_giadung\projects\hql_spark_bridge\output\migration\com_r_m21_countrystate\processing_steps
✅ [Bước 2] Đã xử lý Metadata thành công!
   -> Model nhận diện được: Model 2a
   -> Khóa (Key) nhận diện được: []
   -> File YAML đã lưu tại: C:\Users\ext_giadung\projects\hql_spark_bridge\output\migration\com_r_m21_countrystate\metadata\com_r_m21_countrystate.yaml
 BƯỚC 3: SINH CODE PYSPARK (GENERATOR)
Error rendering DML template model_3a/com_m_dml.jinja: 'key' is undefined
**************************************************
Optimization for node: <class 'sqlglot.expressions.Create'>
node_type: <class 'sqlglot.expressions.Alter'>, False
node_type: <class 'sqlglot.expressions.Drop'>, False
node_type: <class 'sqlglot.expressions.Create'>, True
node_type: 

In [15]:
for table in table_list:
    ddl_file_path = Path(f"C:/Users/ext_giadung/projects/hql_spark_bridge/output/migration/ddl/com/com_t_{table}.sql")
    print(ddl_file_path.read_text())

FileNotFoundError: [Errno 2] No such file or directory: "C:\\Users\\ext_giadung\\projects\\hql_spark_bridge\\output\\migration\\ddl\\com\\com_t_['general_country', '1'].sql"

In [ ]:
dml_context["main_processing_sqls"]

In [ ]:

# Read pre_processing SQLs
pre_processing_sqls = []
for step in pipeline_config.get("pre_processing", []):
    if step.get("action") == "skip":
        continue
    step_file = Path(step["file"])
    if step_file.exists():
        pre_processing_sqls.append(step_file.read_text(encoding="utf-8"))